# Exemplo de Agente com banco vetorial - RAG

Neste exemplo, implementamos um Agente com base vetorial para arquivos de texto. Desde a ingestão, vetorização e inserção no banco, até a implementação da similaridade e transformação disso em uma ferramenta agêntica. Durante o desenvolvimento, serão discutidos evoluções e limitações da implementação. 

## Objetivo Didático

Demonstrar na prática:

- Como estruturar uma base vetorial pgvector
- Conectando com a base
- Ingestão de arquivos com modelo de embeddings
- Adição de técnica de quebra de chunks e overlaping
- Uso de vetorização como janela de contexto da LLM
- Estruturação de um agente RAG

## Parte 1 - Estruturação do banco vetorial

### Instalando biblioteca para comunicar com o banco pgvector

In [1]:
! pip install "psycopg[binary]"


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\Amanda Machado\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Carregando as chaves

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

ip_banco = os.getenv("IP_BANCO")
db_password = os.getenv("DB_PASSWORD")
api_key = os.getenv("OCI_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

### Testar conexão no banco

In [3]:
import psycopg
def get_conn():
    return psycopg.connect(
        host=ip_banco,
        dbname="vetores",
        user="appuser",
        password=db_password,
        port=5432
    )

conn = get_conn()
cur = conn.cursor()
cur.execute("SELECT version();")
print(cur.fetchone())

cur.close()
conn.close()

('PostgreSQL 15.17 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 11.5.0 20240719 (Red Hat 11.5.0-11), 64-bit',)


### Criando a estrutura do banco, se não existir

In [4]:
conn = get_conn()
cur = conn.cursor()

# Criar extensão (caso ainda não exista)
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

# Criar tabela
cur.execute("""
CREATE TABLE IF NOT EXISTS documentos (
    id SERIAL PRIMARY KEY,
    tipo TEXT,
    titulo TEXT,
    conteudo TEXT,
    embedding VECTOR(1536)
);
""")

# Criar índice vetorial
cur.execute("""
CREATE INDEX IF NOT EXISTS documentos_embedding_idx
ON documentos
USING ivfflat (embedding vector_cosine_ops)
WITH (lists = 100);
""")

conn.commit()

cur.close()
conn.close()

print("Banco estruturado com sucesso")

Banco estruturado com sucesso


## Parte 2 - Ingestão, similaridade vetorial e uso como contexto da LLM

### Instalação das bibliotecas

In [5]:
!pip install openai tiktoken langchain-core langchain==1.1.0 langchain-openai==1.1.10


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\Amanda Machado\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Ingestão de texto puro e pequeno no banco

In [6]:
import psycopg
from openai import OpenAI

client = OpenAI(api_key=openai_api_key)

conn = get_conn()

cur = conn.cursor()

texto = "Como redefinir senha do sistema interno..."

embedding = client.embeddings.create(
    model="text-embedding-3-small",
    input=texto
).data[0].embedding

cur.execute(
    """
    INSERT INTO documentos (tipo, titulo, conteudo, embedding)
    VALUES (%s, %s, %s, %s)
    """,
    ("doc", "Reset de senha", texto, embedding)
)

conn.commit()
cur.close()
conn.close()

### Exemplo de busca vetorial simples

In [7]:
pergunta = "Como posso trocar minha senha?"

embedding_query = client.embeddings.create(
    model="text-embedding-3-small",
    input=pergunta
).data[0].embedding

conn = get_conn()
cur = conn.cursor()
cur.execute("""
SELECT titulo, conteudo
FROM documentos
ORDER BY embedding <=> %s::vector
LIMIT 5
""", (embedding_query,))

resultados = cur.fetchall()
cur.close()
conn.close()

### Resultado

In [8]:
resultados

[('Reset de senha', 'Como redefinir senha do sistema interno...'),
 ('Reset de senha', 'Como redefinir senha do sistema interno...'),
 ('Manual - chunk 2',
  's comuns:\n\n- Esqueci minha senha\n- Código 2FA inválido\n- Usuário bloqueado após 5 tentativas\n- Conta inativa por 90 dias sem login\n\nProcedimento de reset de senha:\n\n1. Usuário clica em "Esqueci minha senha"\n2. Sistema envia e-mail automático\n3. Link válido por 30 minutos\n4. Nova senha deve conter:\n   - 8 caracteres mínimo\n   - 1 letra maiúscula\n   - 1 número\n   - 1 caractere especial\n\n============================================================\n3. BLOQUEIO DE CONTA\n=============================='),
 ('Manual - chunk 7',
  'itação de melhoria\nSLA: 72 horas\n\n============================================================\n7. POLÍTICAS DE SEGURANÇA\n============================================================\n\n- Senha deve ser trocada a cada 90 dias\n- 2FA obrigatório\n- Logs mantidos por 180 dias\n- Backup diá

## Parte 2 - Evoluindo a vetorização

### Tecnica de quebra de chunking para arquivos maiores

In [9]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

### Ingestão do arquivo

In [10]:
texto_grande = open("manual.txt", encoding="utf-8").read()
chunks = chunk_text(texto_grande)
conn = get_conn()
cur = conn.cursor()
for i, chunk in enumerate(chunks):
    embedding = client.embeddings.create(
        model="text-embedding-3-small",
        input=chunk
    ).data[0].embedding

    cur.execute(
        """
        INSERT INTO documentos (tipo, titulo, conteudo, embedding)
        VALUES (%s, %s, %s, %s)
        """,
        ("doc", f"Manual - chunk {i}", chunk, embedding)
    )

conn.commit()
cur.close()
conn.close()

### Similaridade como função

In [11]:
def buscar_contexto(pergunta, k=5):
    embedding_query = client.embeddings.create(
        model="text-embedding-3-small",
        input=pergunta
    ).data[0].embedding
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(
        """
        SELECT conteudo
        FROM documentos
        ORDER BY embedding <=> %s::vector
        LIMIT %s
        """,
        (embedding_query, k)
    )

    resultados = cur.fetchall()
    conn.close()
    cur.close()
    return "\n\n".join([r[0] for r in resultados])

### Exemplo de busca

In [12]:
buscar_contexto("Ticket")

'. Testar comunicação com SEFAZ\n4. Reprocessar nota\n\n============================================================\n5. MÓDULO DE ESTOQUE\n============================================================\n\nFuncionalidades:\n\n- Cadastro de produtos\n- Controle de entradas e saídas\n- Alertas de estoque mínimo\n- Inventário automático\n\nProblema comum:\n\n"Estoque negativo"\n\nCausa provável:\n- Venda registrada antes da entrada\n- Ajuste manual incorreto\n\nProcedimento:\n\n1. Verificar histórico de movimentação\n2. Confe\n\n MÓDULO FINANCEIRO\n============================================================\n\nPrincipais funcionalidades:\n\n- Emissão de notas fiscais\n- Controle de contas a pagar\n- Controle de contas a receber\n- Geração de relatórios contábeis\n\nErro comum:\n\n"Erro ao gerar nota fiscal"\n\nCausa provável:\n- Certificado digital vencido\n- Token desconectado\n- Instabilidade na SEFAZ\n\nProcedimento:\n\n1. Verificar validade do certificado\n2. Confirmar conexão do token

## Parte 3 - Utilizando a base vetorial como extensão do contexto da LLM

In [13]:
base_url = "https://inference.generativeai.us-chicago-1.oci.oraclecloud.com/20231130/actions/v1"  # ou sua URL
client_llm = OpenAI(base_url=base_url, api_key=api_key)

def responder(pergunta):
    contexto = buscar_contexto(pergunta)
    #print(contexto)
    prompt = f"""
    Você é um agente de suporte técnico.
    Use apenas o contexto abaixo para responder.

    Contexto:
    {contexto}

    Pergunta:
    {pergunta}

    Regras importantes:
    - Nunca adicione informações que não venham do contexto
    - Caso o contexto venha sem informação relevante, responda que ainda não tem essa informação
    """

    resposta = client_llm.chat.completions.create(
        model="xai.grok-4-fast-non-reasoning",
        messages=[{"role": "user", "content": prompt}]
    )

    return resposta.choices[0].message.content

In [15]:
responder("Problema com 2FA")

'Olá! Sou o agente de suporte técnico. Entendo que você está enfrentando um problema com o 2FA (autenticação em dois fatores). Vamos resolver isso seguindo o protocolo apropriado.\n\n### Descrição do Problema\nVocê não consegue concluir o login devido a erro no código de autenticação em dois fatores.\n\n### Possíveis Causas\n- Código expirado\n- Horário do celular desincronizado\n- Aplicativo autenticador incorreto\n\n### Procedimento de Resolução\n1. Verifique se o horário do celular está configurado automaticamente.\n2. Gere um novo código no aplicativo autenticador.\n3. Evite reutilizar códigos anteriores.\n4. Caso o erro persista, solicite redefinição do 2FA via suporte.\n5. Após redefinição, configure novamente o aplicativo autenticador.\n\nO prazo estimado para resolução é de até 30 minutos. Se precisar de mais ajuda ou se o problema continuar, forneça detalhes adicionais sobre o erro que está vendo. Estou aqui para auxiliar!'

## Parte 4 - Criando um agente de RAG

### Encapando a função de busca vetorial como tool

In [16]:
from langchain.tools import tool

@tool
def buscar_tickets(query: str) -> str:
    """ Realiza busca vetorial na base de documentos de tickets antigos"""
    result = buscar_contexto(query)
    #print(result)
    return result

### Criando o agente com o framework Langchain

In [17]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="xai.grok-4-fast-non-reasoning",
    temperature=0.2,
    max_tokens=500,
    streaming=False,
    api_key=api_key,
    base_url=base_url
)

tools = [
    buscar_tickets
]

react_prompt_template = """Você é um agente de suporte.
    Responda usando apenas o contexto.
    Cite explicitamente a fonte usando o formato [Fonte X].

    Pergunta:
    {pergunta}

    Tools:
    {tools}

    IMPORTANT:
    - Sua resposta deve ter como base apenas o contexto fornecido pela tool, nunca adicione informações que não estejam no contexto
    - Caso não tenha contexto relevante, diz que ainda não possui informações sobre isso na base, mas está em constante atualização
    - Sempre converse com o usuário de maneira gentil e educada

    Begin!
"""

from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=react_prompt_template
)


### [opcional] Função de extração

In [18]:
from langchain_core.messages import AIMessage

def extract_final_output(agent_response):
    final_text = None
    total_tokens = None

    for msg in reversed(agent_response["messages"]):
        if isinstance(msg, AIMessage):

            if msg.content and final_text is None:
                if "Final Answer:" in msg.content:
                    final_text = msg.content.split("Final Answer:",1)[1].strip()
                else:
                    final_text = msg.content.strip()

            if total_tokens is None:
                total_tokens = agent_response["messages"][-1].usage_metadata["total_tokens"]

        if final_text and total_tokens:
            break

    return final_text, total_tokens

### [opcional] Oberservabilidade do uso de tools

In [19]:
import json
from langchain_core.callbacks import BaseCallbackHandler

class CleanAgentCallback(BaseCallbackHandler):
    def on_tool_start(self, serialized, input_str, **kwargs):
        tool_name = serialized.get("name", "unknown_tool")
        print(f"\n TOOL: {tool_name}")
        print(f"   ↳ input: {input_str}")

    def on_tool_end(self, output, **kwargs):
        print("TOOL RESULT (resumo):")

        content = getattr(output, "content", output)

        try:
            data = json.loads(content)
            print("   ↳", data)
            

        except Exception:
            text = str(content)
            print("   ↳", text)


### Teste 1 - Pergunta sobre resete de senha

In [20]:
callback = CleanAgentCallback()

pergunta = "Cliente não consegue acessar o sistema após resetar senha"

response = agent.invoke(
    {"messages": pergunta},
    config={"callbacks": [callback]}
)

final_answer, total_tokens = extract_final_output(response)

print("\nFinal answer:")
print(final_answer)

print("\nMetadados:")
print(f"total tokens: {total_tokens}")


 TOOL: buscar_tickets
   ↳ input: {'query': 'Cliente não consegue acessar o sistema após resetar senha'}
TOOL RESULT (resumo):
   ↳ s comuns:

- Esqueci minha senha
- Código 2FA inválido
- Usuário bloqueado após 5 tentativas
- Conta inativa por 90 dias sem login

Procedimento de reset de senha:

1. Usuário clica em "Esqueci minha senha"
2. Sistema envia e-mail automático
3. Link válido por 30 minutos
4. Nova senha deve conter:
   - 8 caracteres mínimo
   - 1 letra maiúscula
   - 1 número
   - 1 caractere especial

3. BLOQUEIO DE CONTA

Como redefinir senha do sistema interno...

Como redefinir senha do sistema interno...

. BLOQUEIO DE CONTA

A conta é bloqueada automaticamente após:

- 5 tentativas de login inválidas
- Tentativa de acesso suspeita
- Violação de política de segurança

Para desbloquear:

1. Abrir ticket interno
2. Validar identidade do usuário
3. Resetar tentativas via painel administrativo
4. Registrar ocorrência no sistema

4. MÓDULO FINANCEIRO

 integração no painel